In [1]:
import os
import pandas as pd

# points to CLAS folder 
data_path = "D:/Major Project/CLAS/"

In [2]:
blocks = pd.read_csv(data_path + "Block_details/Part1_Block_Details.csv")
print(blocks.shape)
print(blocks.head(20))
print(blocks.columns.tolist())

(37, 9)
    Block  Block Type    ECG File     EDA&PPG File  Length(s)  EDA Quality  \
0       1    Baseline   1_ecg.csv   1_gsr_ppg_.csv      60.02          2.0   
1      10     Neutral  10_ecg.csv  10_gsr_ppg_.csv      29.99          1.0   
2      11  Video clip  11_ecg.csv  11_gsr_ppg_.csv      60.02          2.0   
3      12  Video clip  12_ecg.csv  12_gsr_ppg_.csv      60.03          2.0   
4      13  Video clip  13_ecg.csv  13_gsr_ppg_.csv      60.01          2.0   
5      14  Video clip  14_ecg.csv  14_gsr_ppg_.csv      60.00          2.0   
6      15     Neutral  15_ecg.csv  15_gsr_ppg_.csv      30.01          1.0   
7      16  Video clip  16_ecg.csv  16_gsr_ppg_.csv      60.04          2.0   
8      17  Video clip  17_ecg.csv  17_gsr_ppg_.csv      60.92          2.0   
9      18  Video clip  18_ecg.csv  18_gsr_ppg_.csv      59.11          2.0   
10     19  Video clip  19_ecg.csv  19_gsr_ppg_.csv      60.05          1.0   
11      2   Math Test   2_ecg.csv   2_gsr_ppg_.csv     1

In [3]:
ecg = pd.read_csv(data_path + "Participants/Part1/by_block/1_ecg_.csv")
print(ecg.shape)
print(ecg.head())
print(ecg.columns.tolist())

(15366, 4)
   Timestamp  PythonTimestamp        ecg1       ecg2
0    5657982     1.551792e+09 -605.000072 -28.231723
1    5658110     1.551792e+09 -605.000072 -27.248561
2    5658238     1.551792e+09 -605.000072 -25.665347
3    5658366     1.551792e+09 -605.000072 -25.818966
4    5658494     1.551792e+09 -605.000072 -27.447184
['Timestamp', 'PythonTimestamp', 'ecg1', 'ecg2']


In [4]:
gsr = pd.read_csv(data_path + "Participants/Part1/by_block/1_gsr_ppg_.csv")
print(gsr.shape)
print(gsr.head())
print(gsr.columns.tolist())

(15366, 7)
   Timestamp  PythonTimestamp  accelx  accely  accelz          ppg         gsr
0    8103095     1.551792e+09    1959    2739    2392  1423.443223  399.006139
1    8103223     1.551792e+09    1955    2746    2392  1418.315018  398.540146
2    8103351     1.551792e+09    1956    2744    2393  1412.454212  398.540146
3    8103479     1.551792e+09    1956    2740    2393  1405.128205  398.773006
4    8103607     1.551792e+09    1954    2743    2390  1400.000000  399.239544
['Timestamp', 'PythonTimestamp', 'accelx', 'accely', 'accelz', 'ppg', 'gsr']


In [5]:
print(blocks['Block Type'].unique())
print(blocks['Block Type'].value_counts())

['Baseline' 'Neutral' 'Video clip' 'Math Test' 'Math Test Response'
 'Pictures' 'Stroop Test' 'Stroop Test Response' 'IQ Test'
 'IQ Test Response']
Block Type
Video clip              16
Neutral                 10
Pictures                 4
Baseline                 1
Math Test                1
Math Test Response       1
Stroop Test              1
Stroop Test Response     1
IQ Test                  1
IQ Test Response         1
Name: count, dtype: int64


In [6]:
print(blocks['ECG Quality'].unique())
print(blocks['EDA Quality'].unique())

[2.]
[2. 1.]


In [7]:
# Strip column name spaces
blocks.columns = blocks.columns.str.strip()

In [8]:
# Define your label mapping
label_map = {
    'Baseline'   : 'LOW',
    'Neutral'    : 'LOW',
    'Math Test'  : 'HIGH',
    'IQ Test'    : 'HIGH',
    'Stroop Test': 'MEDIUM'
    # everything else gets dropped
}

In [9]:
# Step 3 - filter only the block types we want
blocks_filtered = blocks[blocks['Block Type'].isin(label_map.keys())]

In [10]:
# Step 4 - assign labels
blocks_filtered = blocks_filtered.copy()
blocks_filtered['Label'] = blocks_filtered['Block Type'].map(label_map)

In [11]:
# Step 5 - filter poor quality
# keep only rows where BOTH ECG and EDA quality == 2.0
blocks_filtered = blocks_filtered[
    (blocks_filtered['ECG Quality'] == 2.0) & 
    (blocks_filtered['EDA Quality'] == 2.0)
]

In [12]:
print(blocks_filtered[['Block', 'Block Type', 'Label', 'ECG File', 'EDA&PPG File']].to_string())
print(blocks_filtered['Label'].value_counts())
print(blocks_filtered.columns)

    Block Block Type Label    ECG File     EDA&PPG File
0       1   Baseline   LOW   1_ecg.csv   1_gsr_ppg_.csv
11      2  Math Test  HIGH   2_ecg.csv   2_gsr_ppg_.csv
23     30    Neutral   LOW  30_ecg.csv  30_gsr_ppg_.csv
25     32    Neutral   LOW  32_ecg.csv  32_gsr_ppg_.csv
27     34    Neutral   LOW  34_ecg.csv  34_gsr_ppg_.csv
29     36    Neutral   LOW  36_ecg.csv  36_gsr_ppg_.csv
34      7    Neutral   LOW   7_ecg.csv   7_gsr_ppg_.csv
35      8    IQ Test  HIGH   8_ecg.csv   8_gsr_ppg_.csv
Label
LOW     6
HIGH    2
Name: count, dtype: int64
Index(['Block', 'Block Type', 'ECG File', 'EDA&PPG File', 'Length(s)',
       'EDA Quality', 'ECG Quality', 'PPG Quality', 'Unnamed: 8', 'Label'],
      dtype='object')


In [13]:
# pick the first block to test with
test_block = blocks_filtered.iloc[0]
print(test_block)

Block                        1
Block Type            Baseline
ECG File             1_ecg.csv
EDA&PPG File    1_gsr_ppg_.csv
Length(s)                60.02
EDA Quality                2.0
ECG Quality                2.0
PPG Quality                2.0
Unnamed: 8                 NaN
Label                      LOW
Name: 0, dtype: object


In [14]:
# load its ECG signal
ecg_file = test_block['ECG File']
gsr_file = test_block['EDA&PPG File']

# add the correct subfolder path
participant_path = data_path + "Participants/Part1/by_block/"

In [15]:
# fix ECG filename mismatch
ecg_file = test_block['ECG File'].replace('1_ecg.csv', '1_ecg_.csv')
gsr_file = test_block['EDA&PPG File']  # this one might be fine already

In [16]:
ecg_signal = pd.read_csv(participant_path + ecg_file)
gsr_signal = pd.read_csv(participant_path + gsr_file)

print("ECG shape:", ecg_signal.shape)
print("GSR shape:", gsr_signal.shape)
print("\nECG head:\n", ecg_signal.head())
print("\nGSR head:\n", gsr_signal.head())

ECG shape: (15366, 4)
GSR shape: (15366, 7)

ECG head:
    Timestamp  PythonTimestamp        ecg1       ecg2
0    5657982     1.551792e+09 -605.000072 -28.231723
1    5658110     1.551792e+09 -605.000072 -27.248561
2    5658238     1.551792e+09 -605.000072 -25.665347
3    5658366     1.551792e+09 -605.000072 -25.818966
4    5658494     1.551792e+09 -605.000072 -27.447184

GSR head:
    Timestamp  PythonTimestamp  accelx  accely  accelz          ppg         gsr
0    8103095     1.551792e+09    1959    2739    2392  1423.443223  399.006139
1    8103223     1.551792e+09    1955    2746    2392  1418.315018  398.540146
2    8103351     1.551792e+09    1956    2744    2393  1412.454212  398.540146
3    8103479     1.551792e+09    1956    2740    2393  1405.128205  398.773006
4    8103607     1.551792e+09    1954    2743    2390  1400.000000  399.239544


In [17]:
pip install neurokit2

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: C:\Users\MRUNMAYEE\AppData\Local\Programs\Python\Python313\python.exe -m pip install --upgrade pip


In [20]:
import neurokit2 as nk
import numpy as np

In [23]:
# we only need ecg2 column
ecg_raw = ecg_signal['ecg2'].values

# sampling rate of shimmer ECG is 128 Hz
sampling_rate = 128

# clean the signal and find R peaks
ecg_cleaned = nk.ecg_clean(ecg_raw, sampling_rate=128)
peaks, info = nk.ecg_peaks(ecg_cleaned, sampling_rate=128)

# calculate HRV features
hrv_features = nk.hrv_time(peaks, sampling_rate=128)
hrv_freq = nk.hrv_frequency(peaks, sampling_rate=128)

print(hrv_features)
print(hrv_freq)

C:\Users\MRUNMAYEE\AppData\Local\Programs\Python\Python313\Lib\site-packages\numpy\lib\_nanfunctions_impl.py:1215: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
C:\Users\MRUNMAYEE\AppData\Local\Programs\Python\Python313\Lib\site-packages\neurokit2\hrv\hrv_time.py:165: RuntimeWarning: Mean of empty slice
  out["MeanNN"] = np.nanmean(rri)
C:\Users\MRUNMAYEE\AppData\Local\Programs\Python\Python313\Lib\site-packages\numpy\lib\_nanfunctions_impl.py:2019: RuntimeWarning: Degrees of freedom <= 0 for slice.
  var = nanvar(a, axis=axis, dtype=dtype, out=out, ddof=ddof,


IndexError: index 0 is out of bounds for axis 0 with size 0